# Day 1
# Data Gathering and Manipulation (NumPy & Pandas)




## Part 1 - Locating and understanding the data

### 1.1 Repository layout and data location

Start by inspecting the repository to find likely data folders (examples: `data/`, `datasets/`, `raw/`, `structures/`). Use the Jupyter file browser or a short shell listing in a terminal cell.

### 1.2 Loading the main dataset

The workshop uses the tmQM redox dataset distributed with the companion `gnnredox/` clone. Start the kernel from the workspace root so `./gnnredox` resolves, or set `REPO_PATH` to the absolute path of your clone.



### Set `REPO_ROOT`

Every notebook in this workshop locates the repository the same way: by walking up
from the notebook's own directory until it finds one containing both `data/` and
`notebooks/`.

Run this cell before any other code cell, and check that the printed path is your clone.


In [ ]:
# --- Set REPO_ROOT --------------------------------------------------------
# Locate the repository root by searching upward from this notebook's directory.
from pathlib import Path


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for d in (start, *start.parents):
        if (d / "data").is_dir() and (d / "notebooks").is_dir():
            return d
    raise FileNotFoundError(f"Repo root not found above {start}")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"
FIG_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

print("Repo root: ", REPO_ROOT)
print("Data dir:  ", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Figure dir:", FIG_DIR)


In [ ]:
import pandas as pd

# All data containing  complexes and redox potentials
DATA_PATH = DATA_DIR / 'tmqm_redox_data_full_data.csv'
df = pd.read_csv(DATA_PATH)
df = df.set_index('csd_code')

# the CSV calls the target 'reduction_pot'; Fe(ii) -> Fe(iii) is an oxidation, so the
# column is renamed here and every notebook downstream uses 'redox_pot'
df = df.rename(columns={'reduction_pot': 'redox_pot'})

print(f'Loaded {len(df)} complexes from: {DATA_PATH}')


# # Mol and smiles of complexes generated after semi-impirically optimizing the geometries solvated exlicitly with water and then desolvating
# # This also represents the gnn-redox dataset
DATA_PATH = DATA_DIR / 'final_df.pkl'
gnn_redox_df = pd.read_pickle(DATA_PATH)


In [ ]:
df

In [ ]:
gnn_redox_df

### 1.3 Filters for the gnn-redox dataset

`df` holds all 2267 complexes from the CSV; `gnn_redox_df` holds the 1546 that survived the
geometry optimization and desolvation, and it is the smaller of the two that defines the
dataset. Before merging them, remove one complex that does not belong.


In [ ]:
print(f'Columns in df = {df.columns.tolist()}') 
print(f'    Shape of DF = {df.shape}')
print(f'-'*40)

# gnn_redox_df.drop(columns=["y"], inplace=True)
print(f'Columns in gnn_redox_df = {gnn_redox_df.columns.tolist()}') 
print(f'    Shape of gnn_redox_df = {gnn_redox_df.shape}')
print(f'-'*40)


In [ ]:
# One complex, SIQHUW, is listed at 7.48 V. The rest of the dataset spans roughly -1.9 to
# 2.5 V, so this sits about ten standard deviations above the mean: a data error rather than
# a chemical extreme. Left in, it would stretch every axis, dominate the correlation
# coefficients, and pull the loss of any model trained on the set.

REDOX_MAX = 5.0

before_df, before_gnn = len(df), len(gnn_redox_df)

print('Complexes above the cutoff:')
print(df.loc[df['redox_pot'] > REDOX_MAX, 'redox_pot'])

df = df[df['redox_pot'] <= REDOX_MAX].copy()
gnn_redox_df = gnn_redox_df[gnn_redox_df['y'] <= REDOX_MAX].copy()

print(f'\ndf:           {before_df} -> {len(df)}')
print(f'gnn_redox_df: {before_gnn} -> {len(gnn_redox_df)}')


### 1.4 Merge dataframes

In [ ]:

# 'y' duplicates redox_pot exactly; drop it so it cannot leak into models
df = pd.concat([df, gnn_redox_df.drop(columns=['y'], errors='ignore')], axis=1, join="inner")


print(f'New Columns in df = {df.columns.tolist()}') 
print(f' New Shape of DF = {df.shape}')

In [ ]:
df


### 1.5 Alternative structural parsing (XYZ-like files)

The structure are represented as ASE atoms objects. The objects are stored in tables in Pickle format, which allows to serialize the python objects.



In [ ]:
# import numpy as np

def parse_distances(atomx):
    # atomx = df1.tmqm_atoms[1]
    for k,v in enumerate(atomx):
        if v.symbol == 'Fe':
            fe_atom = k
            break

    distances = atomx.get_all_distances()[fe_atom]
    distances = np.delete(distances,fe_atom)
    return distances



---

## Part 2 - Visualizing the complexes

### 2.1 Drawing helpers

`show_mol` renders one molecule onto an RDKit Cairo canvas and returns it as a PIL image;
`show_images` tiles a list of those images into a grid.


In [ ]:
from io import BytesIO

from rdkit import Chem
from rdkit.Chem import Draw, rdDepictor
from PIL import Image


def show_mol(d2d, mol, legend='', highlightAtoms=[]):
    d2d.DrawMolecule(mol, legend=legend, highlightAtoms=highlightAtoms)
    d2d.FinishDrawing()
    bio = BytesIO(d2d.GetDrawingText())
    return Image.open(bio)


def show_images(imgs, buffer=0, ncols=5, save_path=None):
    if len(imgs) == 0:
        return None
    rows = [imgs[i:i + ncols] for i in range(0, len(imgs), ncols)]
    row_heights = [max(img.height for img in row) for row in rows]
    row_widths = [sum(img.width for img in row) + buffer * (len(row) - 1) for row in rows]
    total_width = max(row_widths)
    total_height = sum(row_heights) + buffer * (len(rows) - 1)
    res = Image.new("RGBA", (total_width, total_height))
    y = 0
    for row, row_h in zip(rows, row_heights):
        x = 0
        for img in row:
            res.paste(img, (x, y))
            x += img.width + buffer
        y += row_h + buffer
    if save_path:
        res.save(save_path)
    return res



### 2.2 A random sample of 15 complexes

The `mol` column holds RDKit molecule objects carrying the 3D desolvated geometry. Drawing
that conformer directly gives an unreadable projection, so each molecule is copied, stripped
of explicit hydrogens, and given a fresh 2D depiction before rendering. `random_state` fixes
the sample so everyone sees the same complexes.


In [ ]:
SEED = 42
N_SHOW = 15

sample = df.sample(N_SHOW, random_state=SEED)

imgs = []
for csd_code, row in sample.iterrows():
    m = Chem.Mol(row['mol'])        # copy, so the object stored in df is untouched
    m = Chem.RemoveHs(m)            # explicit H dominate the picture
    rdDepictor.Compute2DCoords(m)   # replace the 3D conformer with a flat depiction

    d2d = Draw.MolDraw2DCairo(500, 500)
    dopts = d2d.drawOptions()
    dopts.baseFontSize = 1.0
    dopts.legendFontSize = 30
    imgs.append(show_mol(d2d, m, legend=csd_code))

show_images(imgs, buffer=20, ncols=5)



### 2.3 Interactive 3D view

The 2D depictions above are drawn from a flat layout. `py3Dmol` shows the actual geometry:
drag to rotate, scroll to zoom. The molecules in `mol` already carry 3D coordinates from the
desolvated structures, so no conformer embedding (`AllChem.EmbedMolecule`) is needed.

Change `MOL_INDEX` (0–14) to step through the same 15 complexes.

Atoms are coloured by element (C grey, N blue, O red, S yellow, Cl green, H white, Fe and P
both orange, anything not in the table pink), so every atom is also labelled with its element
symbol. To hide the hydrogen labels, pass `{'not': {'elem': 'H'}}` as the selection instead
of `{}`.

3Dmol discards hydrogens when reading a PDB block, so `addModel` is given `{'keepH': True}`
to keep them.


In [ ]:
import py3Dmol

MOL_INDEX = 0   # 0-14: pick one of the 15 complexes sampled above

csd_code = sample.index[MOL_INDEX]
mol3d = sample['mol'].iloc[MOL_INDEX]
print(f"[{MOL_INDEX}] {csd_code} - {mol3d.GetNumAtoms()} atoms, "
      f"redox_pot = {sample['redox_pot'].iloc[MOL_INDEX]:.3f} V")

pdb_block = Chem.MolToPDBBlock(mol3d)

viewer = py3Dmol.view(width=500, height=500)
viewer.addModel(pdb_block, 'pdb', {'keepH': True})   # 3Dmol strips H from PDB unless asked
viewer.setStyle({'stick': {}, 'sphere': {'radius': 0.4}})
viewer.setStyle({'elem': 'H'}, {'stick': {}, 'sphere': {'radius': 0.2}})   # smaller H

# label every atom with its element symbol
viewer.addPropertyLabels(
    'elem',
    {},
    {'fontSize': 15, 'fontColor': 'black', 'showBackground': False, 'inFront': True},
)

viewer.zoomTo()
viewer.show()



---

## Part 3 - Data inspection with pandas

### 3.1 First look

Show the first few rows and the shape of the data. Use `display()` in Jupyter so wide tables render nicely.



In [ ]:
from IPython.display import display

if 'df' not in globals():
    raise RuntimeError("Load the dataset in the previous cell before running this one.")

print("Shape:", df.shape)
display(df.head())



### 3.2 Data types and summary statistics

Inspect dtypes and a numeric summary. This helps identify columns that need casting or contain unexpected values.



In [ ]:
print(df.dtypes)
display(df.describe(include='all').T)



### 3.3 Column completeness and uniqueness

Check non-null counts and unique values for categorical columns. This gives a quick picture of missingness and label cardinality.



In [ ]:
nulls = df.isnull().sum().sort_values(ascending=False)
print(nulls[nulls>0])

cat_cols = df.select_dtypes(include=['object','category' ]).columns.tolist()

cat_cols.remove('desolv_atoms')
cat_cols.remove('tmqm_atoms')
cat_cols
for c in cat_cols:
    print(c, "-> unique:", df[c].nunique())



### 3.4 Physical meaning of tmQM columns

| Column | Meaning | Notes |
|---|---:|---|
| `csd_code` | Cambridge Structural Database identifier | unique complex identifier |
| `q` | Overall complex charge | integer |
| `Stoichiometry` | Elemental composition | compact formula, e.g. `C12H14FeN4O10` |
| `num_atoms` | Number of atoms | integer |
| `ligands_list` | Serialized ligand SMILES list | supports ligand-count features |
| `tm_oxs` | Transition-metal oxidation state | constant for this Fe(II) subset |
| `ligands_q_list` | Serialized ligand formal charges | supports charge-derived features |
| `redox_pot` | Experimental redox potential | regression target in volts |
| `mol` | RDKit molecule object | built from the desolvated structures; dative bonds to Fe |
| `smiles` | SMILES string | from the desolvated structures; dative-bond notation, e.g. `S=C=[N-]->[Fe+2]...` |
| `desolv_atoms` | ASE `Atoms` 3D geometry | after explicit-water solvation, semi-empirical optimization, and removal of the waters |
| `tmqm_atoms` | ASE `Atoms` 3D geometry | original tmQM geometry; `num_atoms` equals its atom count |
| `y` | Redox potential | same values as `redox_pot` |

### 3.5 Feature engineering from composition and ligands

The raw table has only two useful native numeric predictors (`q` and `num_atoms`); `tm_oxs` is constant. Derive composition and ligand features so PCA and the baseline models have meaningful inputs.



In [ ]:
import ast
import re

TARGET_COL = 'redox_pot'
if TARGET_COL not in df.columns:
    raise RuntimeError(f"Expected target column '{TARGET_COL}' was not found")

def parse_stoichiometry(formula):
    return {
        element: int(count) if count else 1
        for element, count in re.findall(r'([A-Z][a-z]?)(\d*)', formula)
    }

stoichiometry = df['Stoichiometry'].map(parse_stoichiometry)
for element in ['C', 'H', 'N', 'O', 'S', 'Cl', 'P']:
    df[f'n_{element}'] = stoichiometry.map(lambda counts, el=element: counts.get(el, 0))

df['n_ligands'] = df['ligands_list'].map(lambda value: len(ast.literal_eval(value)))
df['q_sum'] = df['ligands_q_list'].map(lambda value: sum(ast.literal_eval(value)))
df['n_anionic'] = df['ligands_q_list'].map(
    lambda value: sum(charge < 0 for charge in ast.literal_eval(value))
)

# constant_cols = [c for c in df.select_dtypes(include='number') if df[c].nunique() <= 1]
# if constant_cols:
#     print('Dropping constant columns:', constant_cols)
#     df = df.drop(columns=constant_cols)

numeric_cols = df.select_dtypes(include='number').columns.tolist()
FEATURE_COLS = [c for c in numeric_cols if c != TARGET_COL]
print('Target:', TARGET_COL)
print('Numeric features:', FEATURE_COLS)



### 3.6 Feature engineering: compute Fe–ligand bond lengths

We compute Euclidean distances between Fe coordinates and ligand atom coordinates. In 3D, the Euclidean distance is:

$$
d_{ij} = \lVert \mathbf{r}_i - \mathbf{r}_j \rVert_2 = \sqrt{\sum_{k=1}^{3} (r_{i,k} - r_{j,k})^2}
$$

Implement a helper that extracts Fe coordinates from a row and computes bond-length statistics (min, max, mean) to be used as features.



In [ ]:
import math
import numpy as np


def compute_fe_bond_lengths(row):
    atomx = row.desolv_atoms
    dists = parse_distances(atomx)
    return pd.Series({'fe_bond_min':  float(np.nanmin(dists)),
                      'fe_bond_mean': float(np.nanmean(dists)),
                      'fe_bond_max':  float(np.nanmax(dists))})


# Apply to the dataframe (may be slow; consider vectorized approaches for large datasets)
# if 'fe_x' in df.columns:
fe_features = df.apply(compute_fe_bond_lengths, axis=1)
df = pd.concat([df, fe_features], axis=1)
print('Computed Fe bond-length features')
# else:
#     print('fe_x/fe_y/fe_z not found; skipping Fe bond-length computation')



---

## Part 4 - Data cleaning

### 4.1 Handling missing values

Strategy: prefer dropping rows with missing target values; for features, either impute (mean/median) or flag missingness via an indicator column.



In [ ]:
# Drop rows missing the target
before = len(df)
df = df[df[TARGET_COL].notnull()].copy()
print(f"Dropped {before - len(df)} rows with missing target '{TARGET_COL}'")

# Impute numeric features with median to be robust to outliers
num_cols = [c for c in df.select_dtypes(include='number') if c != TARGET_COL]
for c in num_cols:
    if df[c].isnull().any():
        med = df[c].median()
        df[f"{c}_was_missing"] = df[c].isnull()
        df[c] = df[c].fillna(med)
        print(f"Imputed {c} with median = {med}")



**Mentor checkpoint 3**: after the data cleaning (Part 4)

- Confirm dataset selection and target column are correct for the group.
- Discuss any domain-specific filters (oxidation state, solvent, basis set) to apply before modeling.
- Decide whether to drop outliers flagged by `is_outlier_any` or to keep them and add robust models later.

Proceed only after confirmation.

---

### Exercise 1 - Quick data exploration

1. Print the distribution (value counts) of a categorical column such as `ligand_type` or `solvent` if present.
2. Report the number of unique Fe centers (hint: group by a unique complex identifier).
3. Compute the fraction of rows that were flagged as outliers.

Write code to answer these questions and print short conclusions (1-2 lines each).

### Exercise 2 - NumPy operations on structure arrays

1. Using the `parse_xyz_file` helper, read one structure file and compute the centroid of its atoms. Use NumPy vector operations.

Recall z-score normalization for a sample:

$$
z_i = \frac{x_i - \mu}{\sigma}
$$

and the sample covariance between X and Y:

$$
\mathrm{Cov}(X,Y) = \frac{1}{n-1}\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})
$$

2. Compute the z-scores for `n_O` and compute its covariance with `redox_pot`.

### Exercise 3 - Save the cleaned dataset

Save `df` to `data_cleaned.csv` in the notebook folder. This file will be used by day2 notebooks.



In [ ]:
# OUTPUT_DIR was created by the REPO_ROOT cell at the top of the notebook

CLEANED_PATH = OUTPUT_DIR / 'data_cleaned.pkl'
df.to_pickle(CLEANED_PATH)#, index=False)
print('Wrote', CLEANED_PATH, 'with', len(df), 'rows')



---

## Summary

| Task | Tool | Key functions |
|---|---:|---|
| Load data | pandas | `pd.read_csv` |
| Inspect | pandas | `df.head`, `df.describe`, `df.dtypes` |
| Outlier detection | pandas, numpy | quantiles, boolean masks |
| Feature engineering | numpy, custom | `compute_fe_bond_lengths` |

---



## Preparation for Day 2
Before tomorrow:
- Ensure your `fe-redox` conda environment is activated and working
- Explore the cloned repo and identify the data files (CSV/NPY/PKL) used by the project
- Brush up on Pandas basics (Pandas 10-minute tutorial: https://pandas.pydata.org/docs/getting_started/10min.html)

We will load and clean the dataset, visualize structure-property relationships, and prepare features for ML models on Day 2. We will cover dimensionality reduction (PCA), clustering (K-Means), and classical ML baselines (Random Forest, Gaussian Process Regression). Make sure `data_cleaned.csv` exists and confirm the shortlist of features you want to try as baselines.